### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="sat11_hand_algo_runtime_iid",
    version_from_unique_name="sat11_hand_algo_runtime",
    version_comment="IID outer splits for the musk dataset",
    # -- Rest as before
    dataset_year="2011",
    domain_str="technology & internet",
    # Data Source
    dataset_source="ASlib",
    original_dataset_source_download_link="https://github.com/coseal/aslib_data/tree/master/SAT11-HAND",
    download_description="""
wget https://raw.githubusercontent.com/coseal/aslib_data/refs/heads/master/SAT11-HAND-ALGO/feature_values.arff \
&& wget https://raw.githubusercontent.com/coseal/aslib_data/refs/heads/master/SAT11-HAND-ALGO/algorithm_feature_values.arff \
&& wget https://raw.githubusercontent.com/coseal/aslib_data/refs/heads/master/SAT11-HAND-ALGO/algorithm_runs.arff \
&& mkdir -p local-data-warehouse/sat11_hand_algo_runtime \
&& mv feature_values.arff algorithm_feature_values.arff algorithm_runs.arff local-data-warehouse/sat11_hand_algo_runtime/
""",
    # References
    academic_reference_bibtex="""@inproceedings{xu-sat12a,
  author    = {L. Xu and F. Hutter and H. Hoos and K. Leyton-Brown},
  title     = {Evaluating Component Solver Contributions to Portfolio-Based Algorithm Selectors},
  pages     = {228-241},
  crossref  = {sat12}
}
@Proceedings{sat12,
  editor =         {A. Cimatti and R. Sebastiani},
  title =         {Proceedings of the Fifteenth International Conference on Theory and Applications of Satisfiability Testing (SAT'12)},
  booktitle = {Proceedings of the Fifteenth International Conference on Theory and Applications of Satisfiability Testing (SAT'12)},
  publisher =         springer,
  series =         lncs,
  volume =         7317,
  year =         2012
}
@article{bischl_aslib_2016,
	title = {{ASlib}: {A} {Benchmark} {Library} for {Algorithm} {Selection}},
	number = {237},
	journal = {Artificial Intelligence Journal (AIJ)},
	author = {Bischl, Bernd and Kerschke, Pascal and Kotthoff, Lars and Lindauer, Marius and Malitsky, Yuri and Fréchette, Alexandre and Hoos, Holger H. and Hutter, Frank and Leyton-Brown, Kevin and Tierney, Kevin and Vanschoren, Joaquin},
	year = {2016},
	pages = {41--58}
}
""",
    academic_reference_bibtex_key="xu-sat12a,sat12,bischl_aslib_2016",
    license="GPLv3",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We get the data from ASlib and merge them into one file.

- Algorithm selection can be solved with many methods (pairwise classification, hierarchical regression, multi-label, ...). We follow the concept from Pulatov et al. (https://proceedings.mlr.press/v188/pulatov22a.html) and learn one single, unified regression model that goes from (instance_features, algorithm_features) -> runtime.
- The description says 'If features are "?", the instance was solved during feature computation.' but from the status file we can see that no task was presovled. So it is unclear what this is referring to. We believe that in all cases we have nan values for features, it is a case of the feature running into a memout, timeout, or crash as stated by the description.
- This regression task is special as it contains censored regression values (capped to some max value) and the model needs to learn this.
- We log scale the target.
- We drop duplicated columns and remove features that leak the target or are not relevant for the predictive task.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="runtime",
    problem_type="regression",
    objective_metric_name="rmse", # better to score with RMSE with cut-off, or like PAR10?
)

## Preprocessing

In [2]:
import arff
import pandas as pd
import uuid
import numpy as np


def load_arff(path) -> pd.DataFrame:
    with open(path, encoding="utf-8") as f:
        data = arff.load(f)
    df = pd.DataFrame(data["data"], columns=[a[0] for a in data["attributes"]])
    return df


df_features = load_arff(dataset_mold.path / "feature_values.arff")
print("feature_values shape:", df_features.shape)
df_algo_features = load_arff(dataset_mold.path / "algorithm_feature_values.arff")
print("algorithm_feature_values shape:", df_algo_features.shape)
df_algo_runs = load_arff(dataset_mold.path / "algorithm_runs.arff")
print("algorithm_runs shape:", df_algo_runs.shape)

# Drop algorithms that for which we do not have features
df_algo_runs = df_algo_runs[df_algo_runs["algorithm"].isin(df_algo_features["algorithm"].unique())]

# Merge
df = df_algo_runs.merge(df_features, on="instance_id", how="left").merge(
    df_algo_features, on="algorithm", how="left"
)
print("Merged data shape:", df.shape)

# Sanitize ID
# Create mapping: molecule -> random string id
mapping = {val: uuid.uuid4().hex[:12] for val in df["instance_id"].unique()}
df["instance_id"] = df["instance_id"].map(mapping)
df["instance_id"] = df["instance_id"].astype("category")

# Drop features
df = df.drop(columns=[
    # Data has no repetitions
    "repetition_x", "repetition_y", "repetition",
    # We want to only rely on algorithm meta features not names.
    "algorithm",
    # Leaks target
    "runstatus",
    # Duplicated columns
    "clustering_min", "edge_ta","edge_to","edge_tl","edge_da","edge_do","edge_dl","edge_as","edge_aa","edge_ao","edge_al","edge_ot","edge_oa","edge_ol","edge_lt","edge_la","edge_ll","degree_min","path_min","VCG_VAR_mean","edge_at",
])

df["runtime"] = np.log(df["runtime"])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

feature_values shape: (296, 117)
algorithm_feature_values shape: (11, 77)
algorithm_runs shape: (4440, 5)
Merged data shape: (2960, 197)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 2,960
Columns: 171
Use sampling: False (sample size: 2,960)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['instance_id', 'gsat_FirstLocalMinStep_Mean', 'gsat_BestAvgImprovement_Mean', 'saps_BestAvgImprovement_Mean', 'saps_FirstLocalMinStep_Mean', 'saps_BestSolution_Mean', 'gsat_BestAvgImprovement_CoeffVariance', 'lobjois_mean_depth_over_vars', 'saps_BestSolution_CoeffVariance', 'gsat_BestSolution_Mean']
Rows remaining as candidates after top-10 filter: 2,960 (of 2,960)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


/home/lennart_priorlabs_ai/.venvs/tabarena_1503/lib/python3.11/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log1p
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [4]:
# Sample Rows
df_head

,instance_id,runtime,nvarsOrig,nclausesOrig,nvars,nclauses,reducedVars,reducedClauses,vars_clauses_ratio,POSNEG_RATIO_CLAUSE_mean,POSNEG_RATIO_CLAUSE_coeff_variation,POSNEG_RATIO_CLAUSE_min,POSNEG_RATIO_CLAUSE_max,POSNEG_RATIO_CLAUSE_entropy,VCG_CLAUSE_mean,VCG_CLAUSE_coeff_variation,VCG_CLAUSE_min,VCG_CLAUSE_max,VCG_CLAUSE_entropy,UNARY,BINARYp,TRINARYp,VCG_VAR_coeff_variation,VCG_VAR_min,VCG_VAR_max,VCG_VAR_entropy,POSNEG_RATIO_VAR_mean,POSNEG_RATIO_VAR_stdev,POSNEG_RATIO_VAR_min,POSNEG_RATIO_VAR_max,POSNEG_RATIO_VAR_entropy,HORNY_VAR_mean,HORNY_VAR_coeff_variation,HORNY_VAR_min,HORNY_VAR_max,HORNY_VAR_entropy,horn_clauses_fraction,VG_mean,VG_coeff_variation,VG_min,VG_max,CG_mean,CG_coeff_variation,CG_min,CG_max,CG_entropy,cluster_coeff_mean,cluster_coeff_coeff_variation,cluster_coeff_min,cluster_coeff_max,cluster_coeff_entropy,DIAMETER_mean,DIAMETER_coeff_variation,DIAMETER_min,DIAMETER_max,DIAMETER_entropy,cl_num_mean,cl_num_coeff_variation,cl_num_min,cl_num_max,cl_num_q90,cl_num_q10,cl_num_q75,cl_num_q25,cl_num_q50,cl_size_mean,cl_size_coeff_variation,cl_size_min,cl_size_max,cl_size_q90,cl_size_q10,cl_size_q75,cl_size_q25,cl_size_q50,SP_bias_mean,SP_bias_coeff_variation,SP_bias_min,SP_bias_max,SP_bias_q90,SP_bias_q10,SP_bias_q75,SP_bias_q25,SP_bias_q50,SP_unconstraint_mean,SP_unconstraint_coeff_variation,SP_unconstraint_min,SP_unconstraint_max,SP_unconstraint_q90,SP_unconstraint_q10,SP_unconstraint_q75,SP_unconstraint_q25,SP_unconstraint_q50,saps_BestSolution_Mean,saps_BestSolution_CoeffVariance,saps_FirstLocalMinStep_Mean,saps_FirstLocalMinStep_CoeffVariance,saps_FirstLocalMinStep_Median,saps_FirstLocalMinStep_Q10,saps_FirstLocalMinStep_Q90,saps_BestAvgImprovement_Mean,saps_BestAvgImprovement_CoeffVariance,saps_FirstLocalMinRatio_Mean,saps_FirstLocalMinRatio_CoeffVariance,gsat_BestSolution_Mean,gsat_BestSolution_CoeffVariance,gsat_FirstLocalMinStep_Mean,gsat_FirstLocalMinStep_CoeffVariance,gsat_FirstLocalMinStep_Median,gsat_FirstLocalMinStep_Q10,gsat_FirstLocalMinStep_Q90,gsat_BestAvgImprovement_Mean,gsat_BestAvgImprovement_CoeffVariance,gsat_FirstLocalMinRatio_Mean,gsat_FirstLocalMinRatio_CoeffVariance,lobjois_mean_depth_over_vars,lobjois_log_num_nodes_over_vars,Lines..Average.,Lines..Total.,Size..Average.,Size..Total.,Number.of.files,Cyclomatic..Average.,Cyclomatic..Total.,Max.Indent..Average.,Max.Indent..Total.,nb_nodes,nb_edges,degree_max,degree_mean,degree_variance,degree_entropy,transitivity,clustering_max,clustering_mean,clustering_variance,paths_max,path_mean,path_variance,path_entropy,Stmt,Type,Decl,Attribute,Operator,Literal,edge_ss,edge_st,edge_sd,edge_sa,edge_so,edge_sl,edge_ts,edge_tt,edge_td,edge_ds,edge_dt,edge_dd,edge_ad,edge_os,edge_od,edge_oo,edge_ls,edge_ld,edge_lo,op_short,op_int,op_long,op_long_long,op_float,op_double,op_bit
0,ca47f6221702,4.090067,2754.0,10377.0,969.0,5950.0,1.8421,0.7440,0.1629,0.4874,0.7382,0.0,1.0,1.8292,0.0044,0.9928,0.0021,0.0289,1.7384,0.0,0.2415,0.5521,3.5252,0.0008,0.2471,2.0945,0.2476,0.2429,0.0000,0.9487,2.1954,0.0016,4.0951,0.0002,0.1696,1.6444,0.5526,0.0046,1.9272,0.0010,0.0595,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.8874,0.0813,3.0,4.0,0.3519,3144.0000,0.0787,2657.0,3485.0,3485.0,2657.0,3337.0,2993.0,3229.0,16.9642,0.1304,14.9793,21.9515,21.9515,14.9793,18.7361,15.7147,16.0341,0.2752,0.9193,0.0003,0.9609,0.6508,0.0283,0.3792,0.0756,0.2055,0.2727,0.3527,0.0000,0.3874,0.3484,0.0842,0.3322,0.2631,0.3076,14.0239,0.2026,180.9720,0.0506,181.0,170.0,193.0,1.3709,0.2342,0.9208,0.0181,16.3016,0.3787,180.8640,0.0592,181.0,169.0,192.0,0.3977,0.9364,0.9249,0.0204,0.1206,0.4990,12.837907,14969.0,10037.734177,792981.0,79.0,2.720930,2691.0,1.731041,1712.0,4334441.0,4545157.0,32037.0,2.097229,259.670236,1.350620,0.000037,0.5,0.002287,0.000368,72.0,6.518183,11.123855,2.433300,0.513657,0.054792,0.352065,0.010225,0.046191,0.023070,3657.0,15.0,453.0,0.0,574.0,0.0,21.0,770.0,205.0,112.0,112.0,3074.0,93.0,347.0,7.0,65.0,146.0,12.0,56.0,0.0,0.25899,0.17986,0.00288,0.00144,0.00576,0.04604
1,

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,instance_id,category,0.0,0.00,296.0,"feba188676d9, 01026dd24a23, 02033e5b65f1, 04418fe8cd39, 05f4dd3c60df, 0700c43bc4a0, 0799ae2bc566, 0814117290c3, da537bb01692, da9d7a59cea7"
1,CG_mean,float64,1810.0,61.15,106.0,"0.026, 0.0198, 0.0174, 0.0206, 0.001, 0.0139, 0.0046, 0.0202, 0.0176, 0.1013"
2,CG_coeff_variation,float64,1810.0,61.15,105.0,"0.8944, 0.2341, 0.6655, 0.5632, 1.1867, 0.3417, 0.2621, 1.0857, 0.0928, 0.0057"
3,CG_min,float64,1810.0,61.15,89.0,"0.0, 0.0003, 0.0001, 0.0021, 0.006, 0.0012, 0.0135, 0.022, 0.0069, 0.0009"
4,CG_max,float64,1810.0,61.15,112.0,"0.0417, 0.1319, 0.0142, 0.0463, 0.0733, 0.1103, 0.0542, 0.0192, 0.2169, 0.0361"
5,CG_entropy,float64,1810.0,61.15,104.0,"0.4506, 0.8158, 4.0244, 4.2844, 0.3622, 2.5663, 2.5202, 0.3906, 0.3034, 0.0437"
6,cluster_coeff_mean,float64,1810.0,61.15,103.0,"0.3492, 0.1766, 0.1958, 0.0766, 0.1983, 0.4461, 0.0545, 0.4474, 0.1046, 0.302"
7,cluster_coeff_coeff_variation,float64,1810.0,61.15,105.0,"0.3252, 0.2063, 0.2489, 0.485, 0.7089, 0.1655, 0.1565, 0.3199, 0.3095, 0.0676"
8,cluster_coeff_min,float64,1810.0,61.15,84.0,"0.0952, 0.1538, 0.3012, 0.3262, 0.4, 0.0014, 0.0065, 0.0095, 0.0435, 0.0556"
9,cluster_coeff_max,float64,1810.0,61.15,70.0,"0.4, 0.2, 0.5, 0.0952, 0.069, 0.0645, 0.9048, 0.8889, 0.1333, 0.0606"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
runtime,2960.0,6.297399e+00,3.867976e+00,-6.908756e+00,8.517193e+00
nvarsOrig,2960.0,2.598159e+03,8.266341e+03,2.000000e+01,6.388200e+04
nclausesOrig,2960.0,9.961375e+04,1.973931e+05,1.050000e+02,1.711065e+06
nvars,2960.0,1.785720e+03,4.741163e+03,2.000000e+01,3.430700e+04
nclauses,2960.0,9.572429e+04,1.925640e+05,1.050000e+02,1.667873e+06
reducedVars,2960.0,3.573963e-01,3.140072e+00,0.000000e+00,5.385140e+01
reducedClauses,2960.0,2.789071e-01,3.213233e+00,0.000000e+00,5.496770e+01
vars_clauses_ratio,2960.0,9.685068e-02,1.290584e-01,6.000000e-04,5.132000e-01
POSNEG_RATIO_CLAUSE_mean,2960.0,7.536118e-01,2.874457e-01,2.437000e-01,1.000000e+00
POSNEG_RATIO_CLAUSE_coeff_variation,2960.0,3.524490e-01,4.109207e-01,0.000000e+00,1.147100e+00


In [7]:
# Categorical Feature Statistics
cat_stats

value  count   pct
column      rank                           
instance_id 1     feba188676d9     10  0.34
            2     01026dd24a23     10  0.34
            3     02033e5b65f1     10  0.34
            4     04418fe8cd39     10  0.34
            5     05f4dd3c60df     10  0.34

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,11.32,-1.741,NaN,14.961,0.355,log1p,15819.8,4.564490e+17,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default IID splits",
    splits=splits
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to sat11_hand_algo_runtime/versions/019dd4a1-d364-7e7e-9ebd-af85a6b33ecb
019dd4a1-d364-7e7e-9ebd-af85a6b33ecb
ef0bf175afa055903ed74130a5e6be0eb7dde475b0e6446aab8958e593fe3eba
